# H-003 · Idiosyncratic Vol Rank

Factor test for **H-003** (equities): whether stocks with **lower** idiosyncratic volatility earn higher next-week returns (idiosyncratic volatility puzzle).

- **Idea** — Rank names by stock-specific return noise after stripping out market exposure (SPY or RSP equal-weight S&P via `BENCHMARK`).
- **Claim** — Low idio-vol rank predicts higher forward returns; high rank predicts lower returns.
- **Why it might work** — Lottery preference and short-sale constraints can leave high idio-vol names overpriced; mispricing is easier to arbitrage in quieter names.
- **Data** — Daily OHLCV long panel + univariate market via `fetch_ohlcv(BENCHMARK)` (`SPY` or `RSP`).

## What is idiosyncratic volatility (`idio_vol`)?

**Idiosyncratic volatility** is the volatility of a stock's returns *after removing market exposure*, not total realised vol.

1. Fit rolling OLS on daily log returns: `r_{i,t} = alpha + beta * r_{mkt,t} + epsilon_t` (`r_mkt` from SPY or RSP)
2. Collect in-window residuals `epsilon_t`
3. **`idio_vol`** = sample std (`ddof=1`) of those residuals over the window (default 20 days)

This is **stock-specific noise** — moves not explained by the market — as opposed to beta-linked risk.

## Raw `idio_vol` vs the H-003 factor signal

| Object | What it is |
|--------|------------|
| **`idio_vol`** (raw) | Rolling residual std from the beta regression; an intermediate datum |
| **H-003 factor** | Cross-sectional percentile rank of `idio_vol` on each date |

The S1 factor panel notebook attaches **beta foundation columns only** (`alpha`, `beta`, `r2`). Idio vol is built here via modular helpers — not inline in the panel notebook — so research, walk-forward, and live code share the same implementation.

- Rolling OLS primitives: `data.processing.feature_implementation.beta`
- Idio vol series/panel helpers: `data.processing.feature_implementation.idiosyncratic_vol`
- Store entrypoint: `add_idio_vol_factors` in `s1_feature_store` with `normalize=True` (CS pct-rank by default — the factor *is* the IVOL rank)

**Per-day residuals (`epsilon_t`):** `epsilon_t = r_{i,t} - alpha_t - beta_t * r_{SPY,t}`. Not stored in the panel by default; `idio_vol` summarizes their dispersion over the window. Per-day residuals could later support abnormal-return event studies, residual momentum, or conditional vol models.

Evaluation uses the S1 **trade-date** panel: Alphalens pivots `open` (no `shift(-1)`); labels are open-to-open. Prior close-to-close ICs are not comparable.


## 0. Imports & Config

Resolve repo root; configure `window` (default 20), `BENCHMARK` (`SPY` or `RSP`), and `normalize` for the eventual store call. Import `fetch_ohlcv`, `fetch_top_n_equities`, `add_idio_vol_factors`, and `market_return_frame`.

In [1]:
import os
import sys

import alphalens as al
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd

from data.ingestion.equity_fetcher import fetch_ohlcv
from data.processing.cleaner import forward_fill_panel
from data.processing.feature_implementation.beta_features import market_return_frame
from data.processing.feature_implementation.gk_vol_ratio import add_realised_vol
from data.processing.feature_implementation.utilities import cross_sectional_pct_rank
from data.processing.s1_feature_store import add_idio_vol_factors

# Jupyter cwd is often this notebook's folder, not the repo root; walk up until we find 01_data/ingestion.
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)

TEARSHEET_DIR = os.path.join(
    ROOT, "02_research", "notebooks", "s1_equities", "factor_tests", "tearsheets"
)

# --- Window screen (edit this list) ---
WINDOWS = [10, 20, 63]  # W — each tested for idio_vol and realised_vol baselines

# --- Fixed for this notebook ---
NORMALIZE = True         # fixed
PERIODS = (1, 5, 21)     # fixed — H-003 / S1 default; primary narrative = 5d
QUANTILES = 5
MAX_LOSS = 0.35
BENCHMARK = "RSP"        # "SPY" (cap-weight) or "RSP" (equal-weight S&P)


## 1. Data Loading

Load daily OHLCV long panel (PIT universe) and univariate market via project fetchers only.

- `fetch_top_n_equities` or saved `s1_factor_panel_train.parquet` for OHLCV (+ optional beta columns)
- `market = fetch_ohlcv(BENCHMARK, ...)` then `market_return_frame(market)` (`SPY` or `RSP`)

In [2]:
panel = pd.read_parquet(TRAIN_PANEL_PATH)
required = {"date", "ticker", "open", "close", "feature_date"}
missing = required - set(panel.columns)
if missing:
    raise ValueError(f"train panel missing columns: {sorted(missing)}")

panel = panel.copy()
panel["date"] = pd.to_datetime(panel["date"])

# Buffer past panel end so Alphalens can compute 21d forward returns near the last IS date.
start = panel["date"].min().strftime("%Y-%m-%d")
end = (panel["date"].max() + pd.Timedelta(days=40)).strftime("%Y-%m-%d")
market = fetch_ohlcv(BENCHMARK, start, end)
market_returns = market_return_frame(market)

print(f"rows={len(panel):,}  tickers={panel['ticker'].nunique():,}  "
      f"dates={panel['date'].nunique():,}  "
      f"[{panel['date'].min().date()} → {panel['date'].max().date()}]")
print(f"{BENCHMARK} market returns: {len(market_returns):,} rows")
panel.head()


rows=289,381  tickers=100  dates=2,915  [2010-01-05 → 2021-08-03]
RSP market returns: 2,942 rows


,date,ticker,open,high,low,close,volume,feature_date,fwd_ret_1,fwd_ret_5,fwd_ret_21
0,2010-01-05,AAPL,6.424143,6.421146,6.357683,6.406478,493729600.0,2010-01-04,-0.001025,-0.025210,-0.083271
1,2010-01-06,AAPL,6.417558,6.453779,6.383729,6.417557,601904800.0,2010-01-05,-0.012268,-0.030367,-0.101456
2,2010-01-07,AAPL,6.338825,6.443003,6.308892,6.315478,552160000.0,2010-01-06,-0.006848,-0.007745,-0.075844
3,2010-01-08,AAPL,6.295419,6.346309,6.258000,6.303801,477131200.0,2010-01-07,0.011888,0.002996,-0.066001
4,2010-01-11,AAPL,6.370258,6.346310,6.258300,6.345711,447610800.0,2010-01-08,-0.016965,-0.021005,-0.079464


## 2. Data Cleaning & Engineering

Build market log returns; attach raw `idio_vol` via `add_idio_vol_factors` (uses beta OLS under the hood).

Any winsorize / floor lives here — the library modules do **not** floor or winsorize by default.

In [3]:
panel = forward_fill_panel(panel, columns=["close"], limit=5)
panel = panel.dropna(subset=["close"]).reset_index(drop=True)
print(f"after clean: rows={len(panel):,}  null close={panel['close'].isna().sum()}")


after clean: rows=289,381  null close=0


## 3. Modeling / Signal Construction

Transform raw `idio_vol` into the H-003 factor: cross-sectional percentile rank within each date.

When implemented in `s1_feature_store`, expect column `idio_vol_rank` (or similar mode name) with `normalize=True` by default.

In [4]:
def add_cs_ranked_realised_vol(
    panel: pd.DataFrame,
    *,
    realised_window: int,
) -> pd.DataFrame:
    """Add CS pct-ranked realised vol as ``realised_vol_{W}`` (baseline)."""
    tmp = f"_realised_vol_tmp_{realised_window}"
    out_col = f"realised_vol_{realised_window}"
    out = add_realised_vol(panel, realised_window=realised_window, col=tmp)
    out[out_col] = cross_sectional_pct_rank(out, tmp)
    return out.drop(columns=[tmp])


In [5]:
panel = add_idio_vol_factors(
    panel,
    market_returns,
    feature_subset=["idio_vol"],
    windows=WINDOWS,
    normalize=NORMALIZE,
)

IDIO_COLS = [c for c in panel.columns if c.startswith("idio_vol")]
REALISED_COLS = []
for w in WINDOWS:
    panel = add_cs_ranked_realised_vol(panel, realised_window=w)
    REALISED_COLS.append(f"realised_vol_{w}")

FACTOR_COLS = IDIO_COLS + REALISED_COLS
print(f"Idio factors ({len(IDIO_COLS)}): {IDIO_COLS}")
print(f"Realised baselines ({len(REALISED_COLS)}): {REALISED_COLS}")


Idio factors (3): ['idio_vol_10', 'idio_vol_20', 'idio_vol_63']
Realised baselines (3): ['realised_vol_10', 'realised_vol_20', 'realised_vol_63']


## 4. Evaluation

Quintile spread and IC of idio-vol rank vs 5-day forward return; compare to total realised vol rank as baseline. Expect negative monotonicity (low rank → long). Use purge/embargo for overlapping 5d labels.

### 4.1 Window screen summary

Factor column names encode kind and window. Each `W` in `WINDOWS` is tested for both idio-vol rank and realised-vol rank.

| Token | Meaning | Formula / role |
|-------|---------|----------------|
| **W** | Rolling window length | Days in the residual-std / realised-vol window ending at `t` |
| `idio_vol_{W}` | H-003 factor | CS pct-rank of residual std vs `BENCHMARK` (SPY or RSP) over W (expect **negative** IC) |
| `realised_vol_{W}` | Baseline | CS pct-rank of close-to-close realised vol over W |

Primary ranking metric below: **mean IC at 5d** (`ic_5d`).


In [6]:
def to_alphalens_prices(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide open matrix for Alphalens (trade-date panel; entry at open)."""
    prices = panel.pivot(index="date", columns="ticker", values="open")
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


def to_alphalens_factor(panel: pd.DataFrame, col: str) -> pd.Series:
    """MultiIndex (date, ticker) factor series for Alphalens."""
    factor = panel.set_index(["date", "ticker"])[col].dropna()
    factor.index = factor.index.set_levels(
        pd.to_datetime(factor.index.levels[0]), level=0
    )
    return factor.sort_index()


def parse_factor_name(col: str) -> dict:
    """Decode ``idio_vol_{W}`` or ``realised_vol_{W}``."""
    if col.startswith("idio_vol_"):
        return {"kind": "idio", "W": int(col.rsplit("_", 1)[-1])}
    if col.startswith("realised_vol_"):
        return {"kind": "realised", "W": int(col.rsplit("_", 1)[-1])}
    if col == "idio_vol":
        return {"kind": "idio", "W": pd.NA}
    if col == "realised_vol":
        return {"kind": "realised", "W": pd.NA}
    raise ValueError(f"unrecognized factor column: {col!r}")


def _period_label(period_index: pd.Index, period: int, position: int):
    """Match Alphalens period label ('1D', '5D', …) or fall back by position."""
    for c in (f"{period}D", f"{period}d", period, str(period)):
        if c in period_index:
            return c
    return period_index[position]


def factor_screen_metrics(
    factor: pd.Series,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
) -> dict:
    """Mean IC and Q5−Q1 mean return spread for each forward period."""
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=factor,
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )
    mean_ic = al.performance.mean_information_coefficient(factor_data)
    mean_ret, _ = al.performance.mean_return_by_quantile(factor_data, demeaned=True)

    row = {}
    for i, p in enumerate(periods):
        ic_key = _period_label(mean_ic.index, p, i)
        ret_key = _period_label(mean_ret.columns, p, i)
        row[f"ic_{p}d"] = float(mean_ic.loc[ic_key])
        q_hi, q_lo = mean_ret.index.max(), mean_ret.index.min()
        row[f"spread_{p}d"] = float(
            mean_ret.loc[q_hi, ret_key] - mean_ret.loc[q_lo, ret_key]
        )
    return row


In [7]:
prices = to_alphalens_prices(panel)

rows = []
for col in FACTOR_COLS:
    meta = parse_factor_name(col)
    metrics = factor_screen_metrics(to_alphalens_factor(panel, col), prices)
    rows.append({"factor": col, **meta, **metrics})

summary = (
    pd.DataFrame(rows)
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)
summary


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

,factor,kind,W,ic_1d,spread_1d,ic_5d,spread_5d,ic_21d,spread_21d
0,idio_vol_10,idio,10,-0.0089,0.0001,-0.0018,0.0009,-0.0006,0.0032
1,idio_vol_20,idio,20,-0.0091,0.0001,-0.0028,0.0010,0.0018,0.0038
2,realised_vol_10,realised,10,-0.0091,0.0001,-0.0032,0.0011,-0.0050,0.0028
3,realised_vol_20,realised,20,-0.0098,0.0002,-0.0044,0.0010,-0.0022,0.0037
4,idio_vol_63,idio,63,-0.0093,0.0001,-0.0048,0.0007,-0.0023,0.0035
5,realised_vol_63,realised,63,-0.0100,0.0002,-0.0061,0.0010,-0.0035,0.0043


### 4.2 Full tear sheet (manual combo)

Review the summary table in §4.1, then set `TEAR_KIND` and `TEAR_WINDOW` in the cell below. The tear sheet runs on `idio_vol_{W}` or `realised_vol_{W}` for those values — nothing is auto-selected.

The tear is displayed in-notebook **and** saved as a multi-page PDF under `02_research/notebooks/s1_equities/factor_tests/tearsheets/` named `H-003_idio_vol_{W}.pdf / H-003_realised_vol_{W}.pdf` (hypothesis id + factor column, including window/mode args). Re-running overwrites the same path.


In [8]:
def tear_factor_col(kind: str, window: int) -> str:
    """Column name for tear sheet: ``idio_vol_{W}`` or ``realised_vol_{W}``."""
    if kind == "idio":
        return f"idio_vol_{window}"
    if kind == "realised":
        return f"realised_vol_{window}"
    raise ValueError(f"kind must be 'idio' or 'realised', got {kind!r}")


def run_full_tear(
    panel: pd.DataFrame,
    factor_col: str,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
    tearsheet_dir: str = TEARSHEET_DIR,
):
    """Build factor_data, run Alphalens full tear, save figs to multi-page PDF.

    Alphalens calls plt.show() after each plot, which clears figures under Agg.
    Temporarily replace plt.show so each figure is written into the PDF before close.
    """
    if factor_col not in panel.columns:
        raise ValueError(
            f"{factor_col!r} not in panel — pick kind/W that were screened "
            f"(available: {FACTOR_COLS})"
        )
    plt.close("all")
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=to_alphalens_factor(panel, factor_col),
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )

    os.makedirs(tearsheet_dir, exist_ok=True)
    out_path = os.path.join(tearsheet_dir, f"H-003_{factor_col}.pdf")
    pdf = PdfPages(out_path)
    n_pages = 0
    _original_show = plt.show

    def _show_and_savefig(*args, **kwargs):
        nonlocal n_pages
        for num in list(plt.get_fignums()):
            fig = plt.figure(num)
            if fig.axes:
                pdf.savefig(fig, bbox_inches="tight")
                n_pages += 1
        plt.close("all")

    plt.show = _show_and_savefig
    try:
        al.tears.create_full_tear_sheet(factor_data, long_short=True)
        _show_and_savefig()
    finally:
        plt.show = _original_show
        pdf.close()
        plt.close("all")

    print(f"Wrote {out_path} ({n_pages} pages)")
    return factor_data


In [9]:
# --- Edit after reviewing the §4.1 summary table ---
TEAR_KIND = "realised"       # idio | realised
TEAR_WINDOW = 63             # W

tear_col = tear_factor_col(TEAR_KIND, TEAR_WINDOW)
print(f"Tear sheet factor: {tear_col}")
tear_factor_data = run_full_tear(panel, tear_col, prices)


Tear sheet factor: realised_vol_63


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Quantiles Statistics


,min,max,mean,std,count,count %
factor_quantile,,,,,,
1,0.0100,0.2062,0.1058,0.0581,56620,20.1464
2,0.2100,0.4040,0.3061,0.0575,55968,19.9144
3,0.4082,0.6020,0.5050,0.0573,55868,19.8788
4,0.6061,0.8000,0.7040,0.0574,55968,19.9144
5,0.8041,1.0000,0.9043,0.0581,56619,20.1460


Returns Analysis


...

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\performance.py:118: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  grouper.append(pd.Grouper(freq=by_time))
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:928: UserWarning: Skipping return periods that aren't exact multiples of days.
  warnings.warn(


Turnover Analysis


,1D,5D,21D
Quantile 1 Mean Turnover,0.0360,0.0940,0.2200
Quantile 2 Mean Turnover,0.0800,0.2020,0.4340
Quantile 3 Mean Turnover,0.0840,0.2120,0.4580
Quantile 4 Mean Turnover,0.0650,0.1720,0.3930
Quantile 5 Mean Turnover,0.0260,0.0720,0.1720


,1D,5D,21D
Mean Factor Rank Autocorrelation,0.9950,0.9780,0.9110


Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\02_research\notebooks\factor_tests\tearsheets\H-003_realised_vol_63.pdf (3 pages)
